# Personne 3 — Calcul de l'indicateur d'accessibilité aux médecins généralistes

Ce notebook combine :

- le voisinage routier calculé par la **personne 1** (couples de communes à moins de 20 min et poids associés) ;
- l'offre et la demande communales calculées par la **personne 2**.

L'indicateur est inspiré de l'**APL / E2SFCA** de la DREES.

**Attention :** il ne s'agit pas d'une reproduction exacte de l'APL officiel, car notre offre est mesurée par un nombre de médecins généralistes et non par leur activité réelle.

## 1. Principe

Pour chaque commune d'offre $j$, on calcule d'abord :

$$R_j = \frac{O_j}{\sum_i D_i \, w_{ij}}$$

avec :

- $O_j$ : nombre de médecins généralistes dans la commune $j$ ;
- $D_i$ : demande de la commune $i$ (population totale **ou** population standardisée selon l'âge) ;
- $w_{ij}$ : poids associé au temps routier entre $i$ et $j$.

Puis, pour chaque commune de résidence $i$ :

$$A_i = \sum_j R_j \, w_{ij}$$

On multiplie ensuite par 100 000 pour obtenir un nombre de médecins généralistes accessibles **pour 100 000 habitants**.

On calcule deux versions :

| Indicateur | Demande $D_i$ |
|---|---|
| `accessibilite_brute` | `population` |
| `accessibilite_ajustee_age` | `demande_standardisee` |

**Propriété de contrôle** : comme les mêmes poids sont utilisés aux deux étapes, on doit avoir
$\sum_i D_i A_i = \sum_j O_j$ : chaque médecin est « distribué » exactement une fois entre les habitants. On le vérifie en section 7.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

## 2. Paramètres et chemins

- Entrées : sorties des personnes 1 et 2 (lecture en `https://` si le fichier est public, ou en `s3://`).
- Sortie : enregistrée en local puis, si `UPLOAD_MINIO = True`, envoyée sur notre bucket MinIO.

In [2]:
# --- Entrées (sorties des personnes 1 et 2, dans le bucket de Mariama) -------
CHEMIN_VOISINAGE = (
    "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
    "voisinage_routier_20min.parquet"
)
CHEMIN_OFFRE_DEMANDE = (
    "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
    "offre_demande_mg_commune.parquet"
)

# --- Sorties ----------------------------------------------------------------
NOM_FICHIER_SORTIE = "indicateur.parquet"

# Dossier personnel (toujours accessible en écriture, et hors du repo GitHub)
DOSSIER_SORTIE = Path.home() / "donnees_projet"
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE_LOCAL = DOSSIER_SORTIE / NOM_FICHIER_SORTIE

UPLOAD_MINIO = True   # envoie le résultat sur ton stockage SSPCloud
CHEMIN_S3_SORTIE = f"emma457/Projet_python/{NOM_FICHIER_SORTIE}"

# --- Paramètres de l'indicateur ---------------------------------------------
ECHELLE = 100_000   # résultat exprimé pour 100 000 habitants

In [3]:
def systeme_fichiers_s3():
    # Connexion à MinIO avec les identifiants du service SSPCloud
    import s3fs
    return s3fs.S3FileSystem(
        client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}
    )


def lire_parquet(chemin):
    # Lit un parquet en local, en https (fichier public) ou en s3:// (MinIO)
    chemin = str(chemin)
    if chemin.startswith("s3://"):
        with systeme_fichiers_s3().open(chemin, "rb") as f:
            return pd.read_parquet(f)
    return pd.read_parquet(chemin)

## 3. Chargement des données

In [4]:
voisinage = lire_parquet(CHEMIN_VOISINAGE)
offre_demande = lire_parquet(CHEMIN_OFFRE_DEMANDE)

print("Voisinage :", voisinage.shape)
print("Offre-demande :", offre_demande.shape)

display(voisinage.head())
display(offre_demande.head())

Voisinage : (1082005, 9)
Offre-demande : (34858, 5)


,CODGEO_origine,LIBGEO_origine,CODGEO_destination,LIBGEO_destination,distance_vol_oiseau_km,distance_route_km,temps_route_min,classe_temps,poids_distance
0,01001,L'Abergement-Clémenciat,01001,L'Abergement-Clémenciat,0.000000,0.0000,0.000000,0-10 min,1.0
1,01001,L'Abergement-Clémenciat,01146,Dompierre-sur-Chalaronne,2.564389,5.2653,7.496667,0-10 min,1.0
2,01001,L'Abergement-Clémenciat,01412,Sulignat,3.257992,5.7110,8.463333,0-10 min,1.0
3,01001,L'Abergement-Clémenciat,01093,Châtillon-sur-Chalaronne,3.379304,6.8761,9.068333,0-10 min,1.0
4,01001,L'Abergement-Clémenciat,01188,Illiat,4.827127,7.2143,9.826667,0-10 min,1.0


,CODGEO,LIBGEO,population,demande_standardisee,nb_medecins_generalistes
0,01001,L'Abergement-Clémenciat,859.0,867.710799,0
1,01002,L'Abergement-de-Varey,273.0,270.430838,0
2,01004,Ambérieu-en-Bugey,15554.0,14951.854488,26
3,01005,Ambérieux-en-Dombes,1917.0,1861.977364,1
4,01006,Ambléon,114.0,124.706348,0


## 4. Contrôles et harmonisation des entrées

Avant de calculer, vérifions que les deux tables « se parlent » :

1. colonnes attendues présentes ;
2. codes communes au même format (texte sur 5 caractères) ;
3. pas de doublons ;
4. pas de valeurs manquantes ou négatives, poids compris entre 0 et 1 ;
5. aucun médecin ni habitant « perdu » parce que sa commune n'est pas dans le voisinage ;
6. chaque commune est voisine d'elle-même (temps 0, poids 1).

In [5]:
# 4.1 Colonnes nécessaires
colonnes_voisinage = {"CODGEO_origine", "CODGEO_destination", "poids_distance"}
colonnes_offre_demande = {
    "CODGEO", "LIBGEO", "population",
    "demande_standardisee", "nb_medecins_generalistes",
}

manquantes_voisinage = colonnes_voisinage - set(voisinage.columns)
manquantes_offre = colonnes_offre_demande - set(offre_demande.columns)

if manquantes_voisinage:
    raise ValueError(f"Colonnes manquantes dans le voisinage : {manquantes_voisinage}")
if manquantes_offre:
    raise ValueError(f"Colonnes manquantes dans offre-demande : {manquantes_offre}")

print("Colonnes nécessaires présentes.")

Colonnes nécessaires présentes.


In [6]:
# 4.2 Codes communes : texte sur 5 caractères ('1001' -> '01001', '1001.0' -> '01001', Corse 2A/2B conservée)
def normaliser_codgeo(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

for col in ["CODGEO_origine", "CODGEO_destination"]:
    voisinage[col] = normaliser_codgeo(voisinage[col])
offre_demande["CODGEO"] = normaliser_codgeo(offre_demande["CODGEO"])

print("Exemples de codes :", offre_demande["CODGEO"].sample(5, random_state=0).tolist())

Exemples de codes : ['01329', '59520', '12216', '27287', '69020']


In [7]:
# 4.3 Doublons
if offre_demande["CODGEO"].duplicated().any():
    raise ValueError("La table offre-demande contient des CODGEO dupliqués.")

n_doublons = voisinage.duplicated(["CODGEO_origine", "CODGEO_destination"]).sum()
print("Couples origine-destination en double :", n_doublons)
if n_doublons:
    voisinage = voisinage.drop_duplicates(["CODGEO_origine", "CODGEO_destination"])

print("Nombre de communes :", offre_demande["CODGEO"].nunique())
print("Nombre de couples origine-destination :", len(voisinage))

Couples origine-destination en double : 0
Nombre de communes : 34858
Nombre de couples origine-destination : 1082005


In [8]:
# 4.4 Valeurs manquantes, négatives, poids
cols_num = ["population", "demande_standardisee", "nb_medecins_generalistes"]

print("Valeurs manquantes :")
print(offre_demande[cols_num].isna().sum())
offre_demande[cols_num] = offre_demande[cols_num].fillna(0)

if (offre_demande[cols_num] < 0).any().any():
    raise ValueError("Valeurs négatives dans offre-demande.")

print(
    f"\nPoids : min = {voisinage['poids_distance'].min():.3f}, "
    f"max = {voisinage['poids_distance'].max():.3f}"
)
print("Répartition des poids :")
print(voisinage["poids_distance"].round(3).value_counts().sort_index())

if not voisinage["poids_distance"].between(0, 1).all():
    raise ValueError("Des poids sortent de [0, 1].")

# Un poids nul n'apporte rien au calcul
voisinage = voisinage[voisinage["poids_distance"] > 0].copy()

Valeurs manquantes :
population                  4
demande_standardisee        0
nb_medecins_generalistes    0
dtype: int64

Poids : min = 0.333, max = 1.000
Répartition des poids :
poids_distance
0.333    541094
0.667    332752
1.000    208159
Name: count, dtype: int64


In [9]:
# 4.5 Communes qui ne trouvent pas leur place dans le voisinage
offre_perdue = offre_demande[
    (offre_demande["nb_medecins_generalistes"] > 0)
    & ~offre_demande["CODGEO"].isin(voisinage["CODGEO_destination"])
]
population_perdue = offre_demande[
    (offre_demande["population"] > 0)
    & ~offre_demande["CODGEO"].isin(voisinage["CODGEO_origine"])
]
codes_voisinage_inconnus = (
    set(voisinage["CODGEO_origine"]) | set(voisinage["CODGEO_destination"])
) - set(offre_demande["CODGEO"])

print(
    f"Communes avec médecins absentes des destinations : {len(offre_perdue)} "
    f"({offre_perdue['nb_medecins_generalistes'].sum():.0f} médecins)"
)
print(
    f"Communes peuplées absentes des origines : {len(population_perdue)} "
    f"({population_perdue['population'].sum():,.0f} habitants)"
)
print(f"Codes du voisinage absents d'offre-demande : {len(codes_voisinage_inconnus)}")

# À regarder en priorité : arrondissements Paris / Lyon / Marseille, communes nouvelles
display(offre_perdue.sort_values("nb_medecins_generalistes", ascending=False).head(10))

Communes avec médecins absentes des destinations : 0 (0 médecins)
Communes peuplées absentes des origines : 0 (0 habitants)
Codes du voisinage absents d'offre-demande : 0


,CODGEO,LIBGEO,population,demande_standardisee,nb_medecins_generalistes


In [10]:
# 4.6 Chaque commune doit être voisine d'elle-même (temps 0 -> poids 1)
deja_presentes = set(
    voisinage.loc[
        voisinage["CODGEO_origine"] == voisinage["CODGEO_destination"],
        "CODGEO_origine",
    ]
)
a_ajouter = sorted(set(offre_demande["CODGEO"]) - deja_presentes)
print("Couples (i, i) ajoutés :", len(a_ajouter))

if a_ajouter:
    voisinage = pd.concat(
        [
            voisinage,
            pd.DataFrame({
                "CODGEO_origine": a_ajouter,
                "CODGEO_destination": a_ajouter,
                "poids_distance": 1.0,
            }),
        ],
        ignore_index=True,
    )

Couples (i, i) ajoutés : 0


In [11]:
# 4.7 Comparaison des deux demandes
total_pop = offre_demande["population"].sum()
total_std = offre_demande["demande_standardisee"].sum()
print(f"Population totale         : {total_pop:,.0f}")
print(f"Demande standardisée      : {total_std:,.0f}")
print(f"Rapport                   : {total_std / total_pop:.3f}")

Population totale         : 67,760,573
Demande standardisée      : 67,760,573
Rapport                   : 1.000
